# One-vs-Rest Metrics for Saved Stack

This notebook loads the saved stack (Random Forest + XGBoost + Extra Trees + SVM with LR meta),
runs inference on the test split, and computes one-vs-rest metrics for each class.

In [8]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score

ROOT = Path('/Users/parvgoyal/IP/Multitaste-Model')
EMB_DIR = ROOT / 'combined_embeddings'
FS_DIR = ROOT / 'fs_results'
TRAIN_PATH = EMB_DIR / 'categorical_all_feature_2197_train.csv'
TEST_PATH = EMB_DIR / 'categorical_all_feature_2197_test.csv'
BUNDLE_PATH = FS_DIR / 'stacked_model random forest + xgboost + extra tress + SVM meta_LR.joblib'
RESULTS_PATH = FS_DIR / 'onevsrest_metrics.csv'

LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
NUM_CLASSES = len(LABEL_COLS)

print('Loading bundle from:', BUNDLE_PATH)

Loading bundle from: /Users/parvgoyal/IP/Multitaste-Model/fs_results/stacked_model random forest + xgboost + extra tress + SVM meta_LR.joblib


In [9]:
bundle = joblib.load(BUNDLE_PATH)
selected_feature_names = bundle['selected_feature_names']
fitted_base_models = bundle['fitted_base_models']
meta_estimator = bundle['meta_estimator']

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

X_train = train_df[selected_feature_names].to_numpy(dtype=np.float32)
y_train = train_df[LABEL_COLS].to_numpy().argmax(axis=1)
X_test = test_df[selected_feature_names].to_numpy(dtype=np.float32)
y_test = test_df[LABEL_COLS].to_numpy().argmax(axis=1)

train_probs = []
test_probs = []
for name, model in fitted_base_models:
    train_probs.append(model.predict_proba(X_train))
    test_probs.append(model.predict_proba(X_test))

train_meta = np.hstack(train_probs)
test_meta = np.hstack(test_probs)

y_proba_train = meta_estimator.predict_proba(train_meta)
y_pred_train = y_proba_train.argmax(axis=1)
y_proba_test = meta_estimator.predict_proba(test_meta)
y_pred_test = y_proba_test.argmax(axis=1)

print('Meta shapes (train/test):', train_meta.shape, test_meta.shape)

Meta shapes (train/test): (13365, 20) (4456, 20)


In [10]:
def binary_metrics(y_true_bin, y_pred_bin):
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).ravel()
    precision = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    recall = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    acc = accuracy_score(y_true_bin, y_pred_bin)
    bacc = 0.5 * (recall + specificity)
    mcc = matthews_corrcoef(y_true_bin, y_pred_bin)
    return {
        'ACC': acc,
        'BACC': bacc,
        'PRE': precision,
        'SPEC': specificity,
        'SENS': recall,
        'F1': f1,
        'MCC': mcc,
    }

rows = []
for class_index, class_name in enumerate(LABEL_COLS):
    train_true_bin = (y_train == class_index).astype(int)
    train_pred_bin = (y_pred_train == class_index).astype(int)
    test_true_bin = (y_test == class_index).astype(int)
    test_pred_bin = (y_pred_test == class_index).astype(int)

    train_metrics = binary_metrics(train_true_bin, train_pred_bin)
    test_metrics = binary_metrics(test_true_bin, test_pred_bin)

    row = {'Class': class_name}
    for key, value in train_metrics.items():
        row[f'Train_{key}'] = round(float(value), 4)
    for key, value in test_metrics.items():
        row[f'Test_{key}'] = round(float(value), 4)
    rows.append(row)

ovr_df = pd.DataFrame(rows)
ovr_df.to_csv(RESULTS_PATH, index=False)
print(f'Saved one-vs-rest metrics to {RESULTS_PATH}')
ovr_df

Saved one-vs-rest metrics to /Users/parvgoyal/IP/Multitaste-Model/fs_results/onevsrest_metrics.csv


,Class,Train_ACC,Train_BACC,Train_PRE,Train_SPEC,Train_SENS,Train_F1,Train_MCC,Test_ACC,Test_BACC,Test_PRE,Test_SPEC,Test_SENS,Test_F1,Test_MCC
0,Sweet,0.9499,0.9585,0.9995,0.9992,0.9178,0.9569,0.9019,0.9215,0.9301,0.9792,0.9709,0.8894,0.9321,0.8449
1,Bitter,0.9968,0.9947,0.9883,0.9978,0.9916,0.9899,0.9880,0.9553,0.9108,0.8710,0.9762,0.8453,0.8580,0.8316
2,Umami,0.9996,0.9998,0.9863,0.9995,1.0000,0.9931,0.9929,0.9946,0.9670,0.9000,0.9965,0.9375,0.9184,0.9158
3,Sour,0.9976,0.9987,0.9730,0.9974,1.0000,0.9863,0.9851,0.9861,0.9736,0.8892,0.9887,0.9584,0.9225,0.9156
4,Undefined,0.9523,0.9702,0.7093,0.9470,0.9935,0.8277,0.8162,0.9123,0.8834,0.5818,0.9209,0.8460,0.6894,0.6556
